# Image sequence-order ablation: significance tests

This notebook contains two separate paired comparisons: **Ordered vs. Shuffled** (Section 3) and **Ordered vs. Mean** (Section 4). Section 5 applies Holm correction to these two primary p-values within the Image modality. Run all cells from top to bottom with `image_test_predictions.csv` beside this notebook.

The statistical procedures, difference direction (Ordered minus alternative), random seed (42), 1,000,000 bootstrap resamples, batch size (10,000), and alpha (0.05) are unchanged from the separate notebooks. Each comparison starts its bootstrap generator with the same original seed.

Image inference uses 378 reenactment clusters, preserving Central and Side together; pooled accuracy still uses all 756 views. The primary test is a two-sided centered paired cluster bootstrap with the original +1 correction.

The one-sample t-tests remain sensitivity checks, and participant summaries remain descriptive. The 95% confidence intervals are unadjusted marginal intervals. Dependence between different reenactments of the same participant is not modeled.

## 1. Shared setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ttest_1samp


PREDICTIONS_PATH = Path("image_test_predictions.csv")

RANDOM_SEED = 42

# Use many resamples for stable final confidence intervals and p-values.
# Resampling is processed in batches below to keep memory usage low.
N_BOOTSTRAP = 1_000_000
BOOTSTRAP_BATCH_SIZE = 10_000

ALPHA = 0.05

from statsmodels.stats.multitest import multipletests
results = {}

## 2. Load and validate predictions

Load the common prediction table once. Both comparisons use the same samples, labels, and participant assignments.

In [2]:
prediction_df = pd.read_csv(PREDICTIONS_PATH)

print(f"Rows:    {len(prediction_df)}")
print(f"Columns: {len(prediction_df.columns)}")

prediction_df.head()

Rows:    756
Columns: 39


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,shuffled_prob_surprise,mean_pred_id,mean_pred,mean_prob_anger,mean_prob_disgust,mean_prob_fear,mean_prob_happiness,mean_prob_neutral,mean_prob_sadness,mean_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.002134,0,Anger,0.553419,0.441173,0.000044,0.000001,1.509095e-10,0.005359,0.000004
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.004542,1,Disgust,0.187334,0.778844,0.000102,0.000002,6.475732e-10,0.033707,0.000013
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000659,5,Sadness,0.014817,0.007378,0.000023,0.000033,1.182091e-05,0.977726,0.000012
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.001852,5,Sadness,0.008396,0.008872,0.000008,0.000003,2.090306e-07,0.982719,0.000002
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.007957,3,Happiness,0.000239,0.000075,0.001177,0.998240,1.545893e-05,0.000182,0.000072


In [3]:
required_columns = {
    "sample_id",
    "reenactment_id",
    "participant_id",
    "camera_index",
    "true_label_id",
    "ordered_pred_id",
    "shuffled_pred_id"
}

missing_columns = required_columns - set(prediction_df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

assert len(prediction_df) == 756
assert prediction_df["sample_id"].is_unique
assert prediction_df["reenactment_id"].nunique() == 378
assert prediction_df["participant_id"].nunique() == 8
assert set(prediction_df["camera_index"].unique()) == {0, 1}

samples_per_reenactment = prediction_df.groupby("reenactment_id").size()
assert samples_per_reenactment.eq(2).all()

views_per_reenactment = prediction_df.groupby("reenactment_id")["camera_index"].nunique()
assert views_per_reenactment.eq(2).all()

true_labels_per_reenactment = prediction_df.groupby("reenactment_id")["true_label_id"].nunique()
assert true_labels_per_reenactment.eq(1).all()

participants_per_reenactment = prediction_df.groupby("reenactment_id")["participant_id"].nunique()
assert participants_per_reenactment.eq(1).all()

print("Prediction table and reenactment/view structure were validated.")

# Both alternatives are required for the within-modality Holm family.
for condition in ["ordered", "shuffled", "mean"]:
    column = f"{condition}_pred_id"
    assert column in prediction_df.columns
    assert prediction_df[column].notna().all()
    assert prediction_df[column].isin(range(7)).all()
assert prediction_df["true_label_id"].isin(range(7)).all()

Prediction table and reenactment/view structure were validated.


### Shared bootstrap function

This is the unchanged resampling function from the separate notebooks; it is called once for each comparison below.

In [4]:
def paired_cluster_bootstrap(differences: np.ndarray,
                             n_bootstrap: int = N_BOOTSTRAP,
                             batch_size: int = BOOTSTRAP_BATCH_SIZE,
                             seed: int = RANDOM_SEED,
                             alpha: float = ALPHA) -> dict:

    differences = np.asarray(differences, dtype=float)

    if differences.ndim != 1 or differences.size == 0:
        raise ValueError("differences must be a nonempty one-dimensional array.")
    if not np.isin(differences, [-1.0, -0.5, 0.0, 0.5, 1.0]).all():
        raise ValueError("Expected paired accuracy differences in {-1, -0.5, 0, 0.5, 1}.")

    n = len(differences)
    observed_difference = differences.mean()

    doubled_differences = (2 * differences).astype(np.int64)
    observed_sum = doubled_differences.sum()

    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=float)
    n_extreme = 0

    for start in range(0, n_bootstrap, batch_size):
        end = min(start + batch_size, n_bootstrap)
        current_batch_size = end - start

        # Ordinary paired bootstrap for the confidence interval.
        bootstrap_indices = rng.integers(0, n, size=(current_batch_size, n))
        bootstrap_means[start:end] = differences[bootstrap_indices].mean(axis=1)

        # Centered-bootstrap null test evaluated in exact integer arithmetic.
        null_indices = rng.integers(0, n, size=(current_batch_size, n))
        null_sums = doubled_differences[null_indices].sum(axis=1)

        n_extreme += np.count_nonzero(
            np.abs(null_sums - observed_sum) >= abs(observed_sum)
        )

    ci_low, ci_high = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])
    p_value = (n_extreme + 1) / (n_bootstrap + 1)

    return {
        "n": n,
        "difference": observed_difference,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "n_bootstrap": n_bootstrap
    }

## 3. Ordered vs. Shuffled — Image

All cells in this section concern **Ordered vs. Shuffled** only. They construct the paired analysis table, verify accuracy, perform the primary test and confidence-interval calculation, and report sensitivity and participant-level results.

### 3.1 Ordered vs. Shuffled: Construct Reenactment-Level Analysis Table

In [5]:
view_df = prediction_df[
    [
        "sample_id",
        "reenactment_id",
        "participant_id",
        "camera_index",
        "true_label_id",
        "ordered_pred_id",
        "shuffled_pred_id"
    ]
].copy()

view_df["ordered_image_correct"] = (
    view_df["ordered_pred_id"] == view_df["true_label_id"]
)

view_df["shuffled_image_correct"] = (
    view_df["shuffled_pred_id"] == view_df["true_label_id"]
)

assert len(view_df) == 756

view_df.head()

,sample_id,reenactment_id,participant_id,camera_index,true_label_id,ordered_pred_id,shuffled_pred_id,ordered_image_correct,shuffled_image_correct
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1,0,0,0,0,True,True
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1,1,0,1,0,False,True
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1,0,5,5,5,True,True
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1,1,5,5,5,True,True
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1,0,3,3,3,True,True


In [6]:
ordered_correct_by_view = (
    view_df
    .pivot(index="reenactment_id", columns="camera_index", values="ordered_image_correct")
    .rename(columns={
        0: "central_ordered_image_correct",
        1: "side_ordered_image_correct"
    })
)

alternative_correct_by_view = (
    view_df
    .pivot(index="reenactment_id", columns="camera_index", values="shuffled_image_correct")
    .rename(columns={
        0: "central_shuffled_image_correct",
        1: "side_shuffled_image_correct"
    })
)

participant_id = view_df.groupby("reenactment_id")["participant_id"].first()
participant_id.name = "participant_id"

analysis_df = ordered_correct_by_view.join(alternative_correct_by_view).join(participant_id)

correctness_columns = [
    "central_ordered_image_correct",
    "side_ordered_image_correct",
    "central_shuffled_image_correct",
    "side_shuffled_image_correct"
]

analysis_df[correctness_columns] = analysis_df[correctness_columns].astype(bool)

analysis_df["ordered_image_correct_mean"] = (
    analysis_df["central_ordered_image_correct"].astype(float)
    + analysis_df["side_ordered_image_correct"].astype(float)
) / 2.0

analysis_df["shuffled_image_correct_mean"] = (
    analysis_df["central_shuffled_image_correct"].astype(float)
    + analysis_df["side_shuffled_image_correct"].astype(float)
) / 2.0

assert len(analysis_df) == 378
assert not analysis_df.isna().any().any()
assert set(analysis_df["ordered_image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})
assert set(analysis_df["shuffled_image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})

analysis_df.head()

,central_ordered_image_correct,side_ordered_image_correct,central_shuffled_image_correct,side_shuffled_image_correct,participant_id,ordered_image_correct_mean,shuffled_image_correct_mean
reenactment_id,,,,,,,
1700478995850-2-1-1-0-0,True,False,True,True,1,0.5,1.0
1700478998549-2-1-1-1-5,True,True,True,True,1,1.0,1.0
1700479001137-2-1-1-2-3,True,False,True,True,1,0.5,1.0
1700479004312-2-1-1-3-0,False,False,True,True,1,0.0,1.0
1700479005401-2-1-1-4-0,True,True,True,True,1,1.0,1.0


### 3.2 Ordered vs. Shuffled: Verify Reported Performance

In [7]:
pooled_ordered_image_accuracy = analysis_df["ordered_image_correct_mean"].mean()
pooled_shuffled_image_accuracy = analysis_df["shuffled_image_correct_mean"].mean()

central_ordered_image_accuracy = analysis_df["central_ordered_image_correct"].mean()
side_ordered_image_accuracy = analysis_df["side_ordered_image_correct"].mean()
central_shuffled_image_accuracy = analysis_df["central_shuffled_image_correct"].mean()
side_shuffled_image_accuracy = analysis_df["side_shuffled_image_correct"].mean()

accuracy_difference = pooled_ordered_image_accuracy - pooled_shuffled_image_accuracy

print(f"Pooled Ordered Image accuracy:  {pooled_ordered_image_accuracy:.4%}")
print(f"Pooled Shuffled Image accuracy: {pooled_shuffled_image_accuracy:.4%}")
print(f"Central Ordered accuracy:       {central_ordered_image_accuracy:.4%}")
print(f"Side Ordered accuracy:          {side_ordered_image_accuracy:.4%}")
print(f"Central Shuffled accuracy:      {central_shuffled_image_accuracy:.4%}")
print(f"Side Shuffled accuracy:         {side_shuffled_image_accuracy:.4%}")
print(f"Ordered - Shuffled:             {100 * accuracy_difference:.2f} percentage points")

assert np.isclose(
    pooled_ordered_image_accuracy,
    (prediction_df["ordered_pred_id"] == prediction_df["true_label_id"]).mean()
)

assert np.isclose(
    pooled_shuffled_image_accuracy,
    (prediction_df["shuffled_pred_id"] == prediction_df["true_label_id"]).mean()
)

assert int((prediction_df["ordered_pred_id"] == prediction_df["true_label_id"]).sum()) == 550
assert np.isclose(pooled_ordered_image_accuracy, 550 / 756)

assert int((prediction_df["shuffled_pred_id"] == prediction_df["true_label_id"]).sum()) == 468
assert np.isclose(pooled_shuffled_image_accuracy, 468 / 756)

Pooled Ordered Image accuracy:  72.7513%
Pooled Shuffled Image accuracy: 61.9048%
Central Ordered accuracy:       73.0159%
Side Ordered accuracy:          72.4868%
Central Shuffled accuracy:      62.9630%
Side Shuffled accuracy:         60.8466%
Ordered - Shuffled:             10.85 percentage points


### 3.3 Ordered vs. Shuffled: Primary Ordered Image vs. Shuffled Image Comparison

The primary estimand is the difference between the reported pooled accuracies,

$\Delta = \mathrm{Accuracy}_{Ordered\ Image} - \mathrm{Accuracy}_{Shuffled\ Image}$.

For each reenactment, the two paired view-level differences are averaged. The mean of these 378 reenactment-level differences is exactly the difference between the two pooled accuracies over all 756 view samples.

The 378 reenactment-level differences are used as the resampling units.

The analysis estimates:

1. a two-sided bootstrap p-value for $H_0: \Delta = 0$, using the centered empirical distribution under the null
2. a percentile-bootstrap 95% confidence interval for $\Delta$

Positive values favor the Ordered Image model.

In [8]:
reenactment_differences = (
    analysis_df["ordered_image_correct_mean"]
    - analysis_df["shuffled_image_correct_mean"]
).to_numpy()

assert np.isclose(reenactment_differences.mean(), accuracy_difference)

primary_result = paired_cluster_bootstrap(
    differences=reenactment_differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    seed=RANDOM_SEED,
    alpha=ALPHA
)

primary_result_df = pd.DataFrame([{
    "comparison": "Pooled Ordered Image - pooled Shuffled Image",
    "n_reenactments": primary_result["n"],
    "ordered_image_accuracy": pooled_ordered_image_accuracy,
    "shuffled_image_accuracy": pooled_shuffled_image_accuracy,
    "difference_pp": 100 * primary_result["difference"],
    "ci_low_pp": 100 * primary_result["ci_low"],
    "ci_high_pp": 100 * primary_result["ci_high"],
    "p_value": primary_result["p_value"],
    "n_bootstrap": primary_result["n_bootstrap"]
}])

primary_result_df

,comparison,n_reenactments,ordered_image_accuracy,shuffled_image_accuracy,difference_pp,ci_low_pp,ci_high_pp,p_value,n_bootstrap
0,Pooled Ordered Image - pooled Shuffled Image,378,0.727513,0.619048,10.846561,7.275132,14.417989,9.999990e-07,1000000


In [9]:
row = primary_result_df.iloc[0]

print(f"Ordered - Shuffled Image accuracy difference: {row['difference_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Two-sided bootstrap p-value: {row['p_value']:.12f}")

Ordered - Shuffled Image accuracy difference: 10.85 percentage points
95% bootstrap CI: [7.28, 14.42] percentage points
Two-sided bootstrap p-value: 0.000000999999


### 3.4 Ordered vs. Shuffled: Sensitivity Check

As a sensitivity analysis, apply a one-sample t-test to the same 378 reenactment-level paired differences.

This is not a separate research hypothesis and will not be included in the multiplicity-correction family. It checks whether the inferential conclusion agrees with the primary bootstrap analysis.

In [10]:
sensitivity_test = ttest_1samp(reenactment_differences, popmean=0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Pooled Ordered Image - pooled Shuffled Image",
    "n_reenactments": len(reenactment_differences),
    "difference_pp": 100 * reenactment_differences.mean(),
    "t_statistic": sensitivity_test.statistic,
    "degrees_of_freedom": sensitivity_test.df,
    "p_value": sensitivity_test.pvalue
}])

sensitivity_result_df

,comparison,n_reenactments,difference_pp,t_statistic,degrees_of_freedom,p_value
0,Pooled Ordered Image - pooled Shuffled Image,378,10.846561,5.87626,377,9.234992e-09


In [11]:
row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['difference_pp']:.2f} percentage points")
print(f"t({row['degrees_of_freedom']:.0f}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")

Mean difference: 10.85 percentage points
t(377) = 5.876
Two-sided sensitivity-check p-value: 0.000000009235


### 3.5 Ordered vs. Shuffled: Participant-Level Heterogeneity Check

This descriptive check examines whether the pooled Ordered-vs.-Shuffled Image difference is directionally consistent across the eight test participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, pooled Ordered Image accuracy, pooled Shuffled Image accuracy, and their difference.

No participant-level significance test or correction factor is applied. The primary inference remains the reenactment-level analysis above; this section is a heterogeneity and plausibility check.

In [12]:
participant_result_df = (
    analysis_df.reset_index()
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        ordered_image_accuracy=("ordered_image_correct_mean", "mean"),
        shuffled_image_accuracy=("shuffled_image_correct_mean", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["ordered_image_accuracy"]
    - participant_result_df["shuffled_image_accuracy"]
)

n_ordered_better = int((participant_result_df["difference_pp"] > 0).sum())
n_alternative_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Ordered Image: {n_ordered_better}/8")
print(f"Participants favoring Shuffled Image: {n_alternative_better}/8")
print(f"Participants tied:                    {n_equal}/8")
print(f"Median participant difference (Ordered - Shuffled Image): {median_participant_difference_pp:.2f} percentage points")

participant_result_df

Participants favoring Ordered Image: 8/8
Participants favoring Shuffled Image: 0/8
Participants tied:                    0/8
Median participant difference (Ordered - Shuffled Image): 12.86 percentage points


,participant_id,n_reenactments,ordered_image_accuracy,shuffled_image_accuracy,difference_pp
0,1,47,0.829787,0.702128,12.765957
1,8,54,0.555556,0.425926,12.962963
2,10,46,0.619565,0.467391,15.217391
3,13,46,0.804348,0.576087,22.826087
4,15,48,0.718750,0.697917,2.083333
5,18,43,0.860465,0.825581,3.488372
6,23,53,0.773585,0.641509,13.207547
7,27,41,0.682927,0.658537,2.439024


### 3.6 Ordered vs. Shuffled: summary and retained results

Retain this comparison’s tables for the joint correction and exports; no test is recalculated.

In [13]:
summary_df = pd.DataFrame({
    "metric": [
        "Pooled Ordered Image accuracy",
        "Pooled Shuffled Image accuracy",
        "Ordered - Shuffled Image difference (pp)",
        "Primary bootstrap CI low (pp)",
        "Primary bootstrap CI high (pp)",
        "Primary bootstrap p-value",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Ordered Image",
        "Participants favoring Shuffled Image",
        "Participants tied",
        "Median participant difference Ordered - Shuffled Image (pp)"
    ],
    "value": [
        pooled_ordered_image_accuracy,
        pooled_shuffled_image_accuracy,
        100 * primary_result["difference"],
        100 * primary_result["ci_low"],
        100 * primary_result["ci_high"],
        primary_result["p_value"],
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_ordered_better,
        n_alternative_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df

,metric,value
0,Pooled Ordered Image accuracy,7.275132e-01
1,Pooled Shuffled Image accuracy,6.190476e-01
2,Ordered - Shuffled Image difference (pp),1.084656e+01
3,Primary bootstrap CI low (pp),7.275132e+00
4,Primary bootstrap CI high (pp),1.441799e+01
5,Primary bootstrap p-value,9.999990e-07
6,Sensitivity t statistic,5.876260e+00
7,Sensitivity t-test p-value,9.234992e-09
8,Participants favoring Ordered Image,8.000000e+00
9,Participants favoring Shuffled Image,0.000000e+00


In [14]:
results["shuffled"] = {
    "reenactment_table": analysis_df.copy(),
    "primary_result": primary_result_df.copy(),
    "sensitivity_ttest": sensitivity_result_df.copy(),
    "participant_level_results": participant_result_df.copy(),
    "summary": summary_df.copy(),
}

## 4. Ordered vs. Mean — Image

All cells in this section concern **Ordered vs. Mean** only. They construct the paired analysis table, verify accuracy, perform the primary test and confidence-interval calculation, and report sensitivity and participant-level results.

### 4.1 Ordered vs. Mean: Construct Reenactment-Level Analysis Table

In [15]:
view_df = prediction_df[
    [
        "sample_id",
        "reenactment_id",
        "participant_id",
        "camera_index",
        "true_label_id",
        "ordered_pred_id",
        "mean_pred_id"
    ]
].copy()

view_df["ordered_image_correct"] = (
    view_df["ordered_pred_id"] == view_df["true_label_id"]
)

view_df["mean_image_correct"] = (
    view_df["mean_pred_id"] == view_df["true_label_id"]
)

assert len(view_df) == 756

view_df.head()

,sample_id,reenactment_id,participant_id,camera_index,true_label_id,ordered_pred_id,mean_pred_id,ordered_image_correct,mean_image_correct
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1,0,0,0,0,True,True
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1,1,0,1,1,False,False
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1,0,5,5,5,True,True
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1,1,5,5,5,True,True
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1,0,3,3,3,True,True


In [16]:
ordered_correct_by_view = (
    view_df
    .pivot(index="reenactment_id", columns="camera_index", values="ordered_image_correct")
    .rename(columns={
        0: "central_ordered_image_correct",
        1: "side_ordered_image_correct"
    })
)

alternative_correct_by_view = (
    view_df
    .pivot(index="reenactment_id", columns="camera_index", values="mean_image_correct")
    .rename(columns={
        0: "central_mean_image_correct",
        1: "side_mean_image_correct"
    })
)

participant_id = view_df.groupby("reenactment_id")["participant_id"].first()
participant_id.name = "participant_id"

analysis_df = ordered_correct_by_view.join(alternative_correct_by_view).join(participant_id)

correctness_columns = [
    "central_ordered_image_correct",
    "side_ordered_image_correct",
    "central_mean_image_correct",
    "side_mean_image_correct"
]

analysis_df[correctness_columns] = analysis_df[correctness_columns].astype(bool)

analysis_df["ordered_image_correct_mean"] = (
    analysis_df["central_ordered_image_correct"].astype(float)
    + analysis_df["side_ordered_image_correct"].astype(float)
) / 2.0

analysis_df["mean_image_correct_mean"] = (
    analysis_df["central_mean_image_correct"].astype(float)
    + analysis_df["side_mean_image_correct"].astype(float)
) / 2.0

assert len(analysis_df) == 378
assert not analysis_df.isna().any().any()
assert set(analysis_df["ordered_image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})
assert set(analysis_df["mean_image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})

analysis_df.head()

,central_ordered_image_correct,side_ordered_image_correct,central_mean_image_correct,side_mean_image_correct,participant_id,ordered_image_correct_mean,mean_image_correct_mean
reenactment_id,,,,,,,
1700478995850-2-1-1-0-0,True,False,True,False,1,0.5,0.5
1700478998549-2-1-1-1-5,True,True,True,True,1,1.0,1.0
1700479001137-2-1-1-2-3,True,False,True,True,1,0.5,1.0
1700479004312-2-1-1-3-0,False,False,True,True,1,0.0,1.0
1700479005401-2-1-1-4-0,True,True,True,True,1,1.0,1.0


### 4.2 Ordered vs. Mean: Verify Reported Performance

In [17]:
pooled_ordered_image_accuracy = analysis_df["ordered_image_correct_mean"].mean()
pooled_mean_image_accuracy = analysis_df["mean_image_correct_mean"].mean()

central_ordered_image_accuracy = analysis_df["central_ordered_image_correct"].mean()
side_ordered_image_accuracy = analysis_df["side_ordered_image_correct"].mean()
central_mean_image_accuracy = analysis_df["central_mean_image_correct"].mean()
side_mean_image_accuracy = analysis_df["side_mean_image_correct"].mean()

accuracy_difference = pooled_ordered_image_accuracy - pooled_mean_image_accuracy

print(f"Pooled Ordered Image accuracy:  {pooled_ordered_image_accuracy:.4%}")
print(f"Pooled Mean Image accuracy: {pooled_mean_image_accuracy:.4%}")
print(f"Central Ordered accuracy:       {central_ordered_image_accuracy:.4%}")
print(f"Side Ordered accuracy:          {side_ordered_image_accuracy:.4%}")
print(f"Central Mean accuracy:      {central_mean_image_accuracy:.4%}")
print(f"Side Mean accuracy:         {side_mean_image_accuracy:.4%}")
print(f"Ordered - Mean:             {100 * accuracy_difference:.2f} percentage points")

assert np.isclose(
    pooled_ordered_image_accuracy,
    (prediction_df["ordered_pred_id"] == prediction_df["true_label_id"]).mean()
)

assert np.isclose(
    pooled_mean_image_accuracy,
    (prediction_df["mean_pred_id"] == prediction_df["true_label_id"]).mean()
)

assert int((prediction_df["ordered_pred_id"] == prediction_df["true_label_id"]).sum()) == 550
assert np.isclose(pooled_ordered_image_accuracy, 550 / 756)

assert int((prediction_df["mean_pred_id"] == prediction_df["true_label_id"]).sum()) == 460
assert np.isclose(pooled_mean_image_accuracy, 460 / 756)

Pooled Ordered Image accuracy:  72.7513%
Pooled Mean Image accuracy: 60.8466%
Central Ordered accuracy:       73.0159%
Side Ordered accuracy:          72.4868%
Central Mean accuracy:      60.3175%
Side Mean accuracy:         61.3757%
Ordered - Mean:             11.90 percentage points


### 4.3 Ordered vs. Mean: Primary Ordered Image vs. Mean Image Comparison

The primary estimand is the difference between the reported pooled accuracies,

$\Delta = \mathrm{Accuracy}_{Ordered\ Image} - \mathrm{Accuracy}_{Mean\ Image}$.

For each reenactment, the two paired view-level differences are averaged. The mean of these 378 reenactment-level differences is exactly the difference between the two pooled accuracies over all 756 view samples.

The 378 reenactment-level differences are used as the resampling units.

The analysis estimates:

1. a two-sided bootstrap p-value for $H_0: \Delta = 0$, using the centered empirical distribution under the null
2. a percentile-bootstrap 95% confidence interval for $\Delta$

Positive values favor the Ordered Image model.

In [18]:
reenactment_differences = (
    analysis_df["ordered_image_correct_mean"]
    - analysis_df["mean_image_correct_mean"]
).to_numpy()

assert np.isclose(reenactment_differences.mean(), accuracy_difference)

primary_result = paired_cluster_bootstrap(
    differences=reenactment_differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    seed=RANDOM_SEED,
    alpha=ALPHA
)

primary_result_df = pd.DataFrame([{
    "comparison": "Pooled Ordered Image - pooled Mean Image",
    "n_reenactments": primary_result["n"],
    "ordered_image_accuracy": pooled_ordered_image_accuracy,
    "mean_image_accuracy": pooled_mean_image_accuracy,
    "difference_pp": 100 * primary_result["difference"],
    "ci_low_pp": 100 * primary_result["ci_low"],
    "ci_high_pp": 100 * primary_result["ci_high"],
    "p_value": primary_result["p_value"],
    "n_bootstrap": primary_result["n_bootstrap"]
}])

primary_result_df

,comparison,n_reenactments,ordered_image_accuracy,mean_image_accuracy,difference_pp,ci_low_pp,ci_high_pp,p_value,n_bootstrap
0,Pooled Ordered Image - pooled Mean Image,378,0.727513,0.608466,11.904762,7.936508,15.873016,9.999990e-07,1000000


In [19]:
row = primary_result_df.iloc[0]

print(f"Ordered - Mean Image accuracy difference: {row['difference_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Two-sided bootstrap p-value: {row['p_value']:.12f}")

Ordered - Mean Image accuracy difference: 11.90 percentage points
95% bootstrap CI: [7.94, 15.87] percentage points
Two-sided bootstrap p-value: 0.000000999999


### 4.4 Ordered vs. Mean: Sensitivity Check

As a sensitivity analysis, apply a one-sample t-test to the same 378 reenactment-level paired differences.

This is not a separate research hypothesis and will not be included in the multiplicity-correction family. It checks whether the inferential conclusion agrees with the primary bootstrap analysis.

In [20]:
sensitivity_test = ttest_1samp(reenactment_differences, popmean=0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Pooled Ordered Image - pooled Mean Image",
    "n_reenactments": len(reenactment_differences),
    "difference_pp": 100 * reenactment_differences.mean(),
    "t_statistic": sensitivity_test.statistic,
    "degrees_of_freedom": sensitivity_test.df,
    "p_value": sensitivity_test.pvalue
}])

sensitivity_result_df

,comparison,n_reenactments,difference_pp,t_statistic,degrees_of_freedom,p_value
0,Pooled Ordered Image - pooled Mean Image,378,11.904762,5.945061,377,6.301333e-09


In [21]:
row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['difference_pp']:.2f} percentage points")
print(f"t({row['degrees_of_freedom']:.0f}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")

Mean difference: 11.90 percentage points
t(377) = 5.945
Two-sided sensitivity-check p-value: 0.000000006301


### 4.5 Ordered vs. Mean: Participant-Level Heterogeneity Check

This descriptive check examines whether the pooled Ordered-vs.-Mean Image difference is directionally consistent across the eight test participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, pooled Ordered Image accuracy, pooled Mean Image accuracy, and their difference.

No participant-level significance test or correction factor is applied. The primary inference remains the reenactment-level analysis above; this section is a heterogeneity and plausibility check.

In [22]:
participant_result_df = (
    analysis_df.reset_index()
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        ordered_image_accuracy=("ordered_image_correct_mean", "mean"),
        mean_image_accuracy=("mean_image_correct_mean", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["ordered_image_accuracy"]
    - participant_result_df["mean_image_accuracy"]
)

n_ordered_better = int((participant_result_df["difference_pp"] > 0).sum())
n_alternative_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Ordered Image: {n_ordered_better}/8")
print(f"Participants favoring Mean Image: {n_alternative_better}/8")
print(f"Participants tied:                    {n_equal}/8")
print(f"Median participant difference (Ordered - Mean Image): {median_participant_difference_pp:.2f} percentage points")

participant_result_df

Participants favoring Ordered Image: 8/8
Participants favoring Mean Image: 0/8
Participants tied:                    0/8
Median participant difference (Ordered - Mean Image): 12.36 percentage points


,participant_id,n_reenactments,ordered_image_accuracy,mean_image_accuracy,difference_pp
0,1,47,0.829787,0.702128,12.765957
1,8,54,0.555556,0.490741,6.481481
2,10,46,0.619565,0.391304,22.826087
3,13,46,0.804348,0.684783,11.956522
4,15,48,0.718750,0.677083,4.166667
5,18,43,0.860465,0.651163,20.930233
6,23,53,0.773585,0.632075,14.150943
7,27,41,0.682927,0.658537,2.439024


### 4.6 Ordered vs. Mean: summary and retained results

Retain this comparison’s tables for the joint correction and exports; no test is recalculated.

In [23]:
summary_df = pd.DataFrame({
    "metric": [
        "Pooled Ordered Image accuracy",
        "Pooled Mean Image accuracy",
        "Ordered - Mean Image difference (pp)",
        "Primary bootstrap CI low (pp)",
        "Primary bootstrap CI high (pp)",
        "Primary bootstrap p-value",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Ordered Image",
        "Participants favoring Mean Image",
        "Participants tied",
        "Median participant difference Ordered - Mean Image (pp)"
    ],
    "value": [
        pooled_ordered_image_accuracy,
        pooled_mean_image_accuracy,
        100 * primary_result["difference"],
        100 * primary_result["ci_low"],
        100 * primary_result["ci_high"],
        primary_result["p_value"],
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_ordered_better,
        n_alternative_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df

,metric,value
0,Pooled Ordered Image accuracy,7.275132e-01
1,Pooled Mean Image accuracy,6.084656e-01
2,Ordered - Mean Image difference (pp),1.190476e+01
3,Primary bootstrap CI low (pp),7.936508e+00
4,Primary bootstrap CI high (pp),1.587302e+01
5,Primary bootstrap p-value,9.999990e-07
6,Sensitivity t statistic,5.945061e+00
7,Sensitivity t-test p-value,6.301333e-09
8,Participants favoring Ordered Image,8.000000e+00
9,Participants favoring Mean Image,0.000000e+00


In [24]:
results["mean"] = {
    "reenactment_table": analysis_df.copy(),
    "primary_result": primary_result_df.copy(),
    "sensitivity_ttest": sensitivity_result_df.copy(),
    "participant_level_results": participant_result_df.copy(),
    "summary": summary_df.copy(),
}

## 5. Holm correction — both Image primary tests

The family contains **Ordered vs. Shuffled** and **Ordered vs. Mean**. Use the two primary p-values already calculated above. Holm controls family-wise error at 0.05 within this modality. Sensitivity tests and participant summaries are excluded; confidence intervals remain unchanged.

When there are zero extreme bootstrap draws, the raw Image p-value is the Monte Carlo floor `1 / (N_BOOTSTRAP + 1)`, not a precisely resolved tail probability. Holm adjusts this reported estimate.

In [25]:
holm_family_df = pd.DataFrame({
    "alternative": ["shuffled", "mean"],
    "p_value": [results[a]["primary_result"].loc[0, "p_value"] for a in ["shuffled", "mean"]],
})
reject, p_holm, _, _ = multipletests(holm_family_df["p_value"], alpha=ALPHA, method="holm")
holm_family_df["p_value_holm"] = p_holm
holm_family_df["reject_holm"] = reject
holm_family_df

,alternative,p_value,p_value_holm,reject_holm
0,shuffled,9.999990e-07,0.000002,True
1,mean,9.999990e-07,0.000002,True


## 6. Export both comparisons

Keep the original five result-CSV filenames per comparison. Add raw and adjusted p-values to the primary results and adjusted values to the summaries. Export one common Holm-family table for this modality.

In [26]:
OUTPUT_DIR = Path("statistical_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for row in holm_family_df.itertuples(index=False):
    tables = results[row.alternative]
    primary = tables["primary_result"]
    primary["p_value_holm"] = row.p_value_holm
    primary["reject_holm"] = row.reject_holm
    primary["holm_family"] = "image_sequence_order"
    primary["holm_family_size"] = 2
    primary["alpha"] = ALPHA
    summary = pd.concat([tables["summary"], pd.DataFrame({
        "metric": ["Primary Holm-adjusted p-value (within modality)", "Reject primary null after Holm correction", "Holm family size", "Family-wise alpha"],
        "value": [row.p_value_holm, int(row.reject_holm), 2, ALPHA],
    })], ignore_index=True)
    for name, table in tables.items():
        if name == "summary":
            table = summary
        table.to_csv(OUTPUT_DIR / f"ordered_image_vs_{row.alternative}_image_{name}.csv",
                     index=True and name == "reenactment_table")

holm_family_df.to_csv(OUTPUT_DIR / "image_sequence_order_holm_family.csv", index=False)
print(f"Results written to: {OUTPUT_DIR.resolve()}")

Results written to: /workspace/repos/emohevrdb-dfer/5_dynamic_facial_expression_recognition/5_1_image_sequence_based_fer/5_1_4_sequence_order_ablation/significance-tests/statistical_results
